# チャレンジ：データサイエンスに関するテキストの分析

この例では、伝統的なデータサイエンスのプロセスのすべてのステップを網羅したシンプルな演習を行います。コードを書く必要はなく、以下のセルをクリックして実行し、結果を観察するだけで構いません。チャレンジとして、異なるデータでこのコードを試してみることをお勧めします。

## 目標

このレッスンでは、データサイエンスに関連するさまざまな概念について議論してきました。<strong>テキストマイニング</strong>を行うことで、さらに関連する概念を発見してみましょう。データサイエンスに関するテキストからキーワードを抽出し、その結果を視覚化してみます。

使用するテキストは、WikipediaのData Scienceのページです：


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## ステップ1: データの取得

すべてのデータサイエンスプロセスの最初のステップはデータを取得することです。ここでは `requests` ライブラリを使います:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## ステップ 2: データの変換

次のステップは、処理に適した形式にデータを変換することです。今回の場合、ページからHTMLのソースコードをダウンロードしているので、それをプレーンテキストに変換する必要があります。

これにはさまざまな方法があります。ここでは、HTML解析のための人気のPythonライブラリである [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/) を使用します。BeautifulSoupを使うと特定のHTML要素をターゲットにできるので、Wikipediaの主要な記事コンテンツに絞り込み、ナビゲーションメニュー、サイドバー、フッターなどの不要なコンテンツを減らすことができます（ただし、一部の定型文は残る場合があります）。


まず、HTML解析のためにBeautifulSoupライブラリをインストールする必要があります：


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## ステップ 3: インサイトの取得

最も重要なステップは、データを洞察を得られる形に変換することです。今回の場合、テキストからキーワードを抽出し、どのキーワードがより意味があるかを確認したいと思います。

キーワード抽出には [RAKE](https://github.com/aneesha/RAKE) というPythonライブラリを使用します。まず、このライブラリがインストールされていない場合はインストールしましょう: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

主な機能は `Rake` オブジェクトから利用でき、いくつかのパラメーターを使ってカスタマイズできます。ここでは、キーワードの最小長さを5文字、ドキュメント内でのキーワードの最小出現回数を3、キーワード内の最大単語数を2に設定します。他の値でも試して結果を観察してみてください。


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


重要度の度合いとともに用語のリストを取得しました。ご覧のとおり、機械学習やビッグデータのような最も関連性の高い分野がリストの上位に存在しています。

## ステップ4：結果の可視化

人は視覚的な形でデータを最もよく解釈できます。したがって、洞察を得るためにデータを可視化することがよく意味を持ちます。Pythonの`matplotlib`ライブラリを使って、キーワードとそれらの関連度の単純な分布をプロットできます：


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

しかし、単語の頻度を可視化するより良い方法があり、それが<strong>ワードクラウド</strong>を使う方法です。キーワードのリストからワードクラウドをプロットするために、別のライブラリをインストールする必要があります。


In [ ]:
!{sys.executable} -m pip install wordcloud

`WordCloud` オブジェクトは、元のテキストか事前に計算された単語とその頻度のリストのいずれかを受け取り、画像を返します。この画像は `matplotlib` を使って表示できます：


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

元のテキストを `WordCloud` に渡すこともできます — 類似の結果が得られるか見てみましょう:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

これでワードクラウドはより印象的になりましたが、多くのノイズ（例：`Retrieved on` のような無関係な単語）も含まれています。また、*data scientist* や *computer science* のような二語から成るキーワードが少なくなっています。これはRAKEアルゴリズムがテキストから良質なキーワードを選択するのがずっと優れているためです。この例は、データの前処理とクリーニングの重要性を示しており、最終的に明瞭な結果が得られることでより良い意思決定が可能になることを示しています。

この演習では、Wikipediaのテキストから意味を抽出し、キーワードやワードクラウドの形で表現する簡単なプロセスを実施しました。この例は非常にシンプルですが、データサイエンティストがデータを扱う際に行う典型的なすべてのステップを、データ取得から可視化までよく示しています。

このコースでは、それらのステップすべてを詳細に議論します。 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
